# 01.3 Autograd

Autograd is one of the core capabilities of PyTorch. If you understand this notebook, `loss.backward()` will stop being a line you merely copy and will become a concrete operation: it traces how the loss depends on tensors that require gradients and stores derivatives in their `.grad` fields.

The important ideas are `requires_grad`, `backward()`, gradient accumulation, `torch.no_grad()`, and `detach()`.

## Learning Goals

After this notebook, you should be able to:

1. Understand what `requires_grad=True` means.
2. Read a simple computational graph.
3. Use `backward()` to compute gradients.
4. Understand why gradients accumulate.
5. Use `torch.no_grad()` correctly.
6. Understand what `detach()` does.

In [ ]:
import torch

## `requires_grad` and the Computational Graph

When a tensor has `requires_grad=True`, `PyTorch` starts tracking operations involving it.

These connected operations form the computational graph.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("x =", x)
print("y =", y)
print("x.requires_grad =", x.requires_grad)
print("y.requires_grad =", y.requires_grad)
print("y.grad_fn =", y.grad_fn)

A useful first intuition is that autograd records how a result was computed from tensors that require gradients. If `x.requires_grad` is true and you compute `y` from `x`, PyTorch keeps enough history to later compute how `y` changes with respect to `x`. The `grad_fn` shown on `y` is a sign that PyTorch is tracking the operation that produced it.

## Using `backward()` to Compute Gradients

If the output is a scalar, you can call `backward()` directly.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1
y.backward()

print("x.grad =", x.grad)

Manual check:

- `y = x^2 + 3x + 1`
- `dy/dx = 2x + 3`

So `x.grad == 7`.


In [ ]:
# Exercise 1
#
# Compute the derivative of y = 4x^2 - x at x = 3 using autograd.
#
# Steps:
# - Create x as a scalar tensor with value 3.0 and requires_grad=True.
# - Compute y = 4 * x ** 2 - x.
# - Call y.backward() to populate x.grad.
# - Print x.grad.
#
# Manual check:
# dy/dx = 8x - 1, so at x = 3 the answer should be 23.

# x =
# y =
# y.backward()
# print(x.grad)

In [ ]:
# Exercise 1 Reference Solution

x = torch.tensor(3.0, requires_grad=True)
y = 4 * x ** 2 - x
y.backward()
print(x.grad)

# manually: dy/dx = 8x - 1, at x=3 => 23

## Non-Scalar Outputs

If the output is not a scalar, `backward()` needs an additional gradient argument.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2

y.backward(gradient=torch.ones_like(y))
print("x.grad =", x.grad)

Passing `torch.ones_like(y)` is equivalent to weighting each output term equally.

In practice, the training `loss` is usually already a scalar, so the common pattern is simply:

- `loss.backward()`

## Gradient Accumulation

This is a very important, very common, and very easy-to-miss point.

In `PyTorch`, gradients accumulate in `.grad` by default.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("after first backward / after first backward:", x.grad)

y2 = 3 * x
y2.backward()
print("after second backward / after second backward:", x.grad)

The gradient becomes `7` because PyTorch accumulates gradients into `.grad` instead of replacing them automatically. The first backward call contributes `d(x^2)/dx = 2x = 4` at `x=2`. The second backward call contributes `d(3x)/dx = 3`. Since the old gradient was not cleared, the stored result becomes `4 + 3 = 7`.

This is the reason training loops call `optimizer.zero_grad()` before the next backward pass.

In [ ]:
# Exercise 2
#
# Reproduce gradient accumulation.
#
# Steps:
# - Create x = 1.0 with requires_grad=True.
# - Compute y = x ** 3 and call y.backward().
# - Then compute z = 2 * x and call z.backward() without clearing x.grad.
# - Print x.grad.
#
# Expected idea:
# - The first backward contributes 3.
# - The second backward contributes 2.
# - The final accumulated gradient should be 5.

# x =
# y =
# y.backward()
# z =
# z.backward()
# print(x.grad)

In [ ]:
# Exercise 2 Reference Solution

x = torch.tensor(1.0, requires_grad=True)
y = x ** 3
y.backward()
z = 2 * x
z.backward()
print(x.grad)

# dy/dx = 3, dz/dx = 2, total = 5

## Manually Zeroing Gradients

Before updating parameters, you usually clear old gradients first.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
print("before zeroing / before zeroing:", x.grad)

x.grad.zero_()
print("after zeroing / after zeroing:", x.grad)

z = 3 * x
z.backward()
print("after backward again / after backward again:", x.grad)

## 6. `torch.no_grad()`

Sometimes you want to run tensor operations without building a gradient graph. This is common during inference, validation, or simple numeric inspection. `torch.no_grad()` tells PyTorch not to track operations inside the block, which reduces memory use and avoids accidental gradient computation.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x * 5

print("y =", y)
print("y.requires_grad =", y.requires_grad)

Main value of `no_grad()`:

- saves graph overhead and memory
- avoids unnecessary gradient tracking

## 7. `detach()`

`detach()` returns a new tensor view that shares data but no longer participates in the current graph.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * x
z = y.detach()

print("y.requires_grad =", y.requires_grad)
print("z.requires_grad =", z.requires_grad)

y_sum = y.sum()
y_sum.backward()
print("x.grad =", x.grad)

Intuitively, you can think of it as:

- `y` is still connected to the graph
- `z` has been cut away from the graph

In [ ]:
# Exercise 3
#
# Decide which tensors below will track gradients.
#
# Read the code first and predict the three printed values:
# - a is computed from x while gradient tracking is enabled.
# - b is computed inside torch.no_grad().
# - c is created with detach(), which breaks the connection to the graph.
#
# Then uncomment and run the code to check your prediction.
#
# x = torch.tensor(2.0, requires_grad=True)
# a = x * 2
# with torch.no_grad():
#     b = x * 3
# c = a.detach()
#
# print(a.requires_grad)
# print(b.requires_grad)
# print(c.requires_grad)

In [ ]:
# Exercise 3 Reference Solution

x = torch.tensor(2.0, requires_grad=True)
a = x * 2
with torch.no_grad():
    b = x * 3
c = a.detach()

print(a.requires_grad)
print(b.requires_grad)
print(c.requires_grad)

## A Tiny Training-Flavored Example

The example below is not yet a full training loop, but it is already very close.


In [ ]:
w = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(2.0)
target = torch.tensor(4.0)

pred = w * x
loss = (pred - target) ** 2
loss.backward()

print("pred =", pred.item())
print("loss =", loss.item())
print("w.grad =", w.grad.item())

The core training chain has already appeared in miniature. A parameter such as `w` is used in a forward calculation to produce a prediction. The prediction is compared with the target to produce a loss. Calling `loss.backward()` computes gradients for the parameter. Later, the full training loop will add batching, an optimizer step, evaluation mode, and metric tracking around this same chain.

## Summary

You should now be able to answer:

1. What exactly does `requires_grad=True` mean?
2. Why can scalar outputs call `backward()` directly?
3. Why does `.grad` accumulate?
4. What is the difference between `no_grad()` and `detach()`?
5. Why do we usually clear gradients before training steps?

Suggested next step:

- `DataLoader`, or directly into `nn.Module` and the training loop.